# Project #2 - Customer Support Ticket Resolver
**Brief from Paras (assignment 2 of 10):**

> Watches an inbox or Zendesk queue for new tickets, classifies intent, searches the company's
> Notion knowledge base for relevant docs, drafts a reply, and either sends it for low-risk tickets
> or posts to Slack for human approval on anything sensitive.

This is the **easiest first** per Paras - only 3 workers, clean conditional branch, real production demand.

## Architecture

```
INCOMING EMAIL
    │
    ▼
SUPERVISOR  --pick worker--
    │       ┌─ classifier_worker  (LLM only, no tools)
    │       ├─ kb_worker          (Notion: query knowledge base)
    │       └─ reply_worker       (Gmail: draft + send)
    │
    │   IF reply is high-risk OR low confidence:
    │      INTERRUPT - post to Slack for human approval
    │
    ▼
END (reply sent or escalated)
```


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect Gmail + Notion + Slack"
print("env OK")


## 2. State - includes ticket-specific fields

In [ ]:
from typing import TypedDict, Annotated, Literal, Optional
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class TicketState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    # Ticket-specific
    ticket_id: str
    ticket_subject: str
    ticket_body: str
    intent: Optional[str]            # billing | technical | account | other
    severity: Optional[str]          # low | medium | high
    kb_excerpt: Optional[str]        # what we found in Notion
    draft_reply: Optional[str]
    needs_human: bool                # True -> escalate to Slack


## 3. LLM and Composio toolkits

In [ ]:
from langchain_openai import ChatOpenAI
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()

# Each worker is scoped to one toolkit
notion_tools = toolset.get_tools(actions=[
    Action.NOTION_QUERY_DATABASE,
    Action.NOTION_FETCH_DATA,
])
gmail_tools = toolset.get_tools(actions=[
    Action.GMAIL_REPLY_TO_EMAIL,
    Action.GMAIL_SEND_DRAFT,
])
slack_tools = toolset.get_tools(actions=[
    Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL,
])


## 4. Worker A - Classifier (no Composio tools, just an LLM call)

In [ ]:
CLASSIFIER_PROMPT = '''You are a support ticket classifier. Read the ticket and output JSON:
{"intent": "billing|technical|account|other", "severity": "low|medium|high"}

Severity rubric:
- high: refund > $500, security/data issue, account locked, angry customer
- medium: feature broken, can't complete a task, refund $50-500
- low: how-to question, minor cosmetic bug, refund < $50

Output JSON only. No prose.'''

import json
def classifier_worker(state: TicketState) -> dict:
    out = llm.invoke([
        SystemMessage(content=CLASSIFIER_PROMPT),
        HumanMessage(content=f"Subject: {state['ticket_subject']}\n\nBody: {state['ticket_body']}"),
    ])
    try:
        parsed = json.loads(out.content.strip())
    except json.JSONDecodeError:
        parsed = {"intent": "other", "severity": "medium"}
    return {
        "intent": parsed.get("intent", "other"),
        "severity": parsed.get("severity", "medium"),
        "messages": [AIMessage(content=f"Classified as {parsed['intent']} / {parsed['severity']}", name="classifier")],
    }


## 5. Worker B - Knowledge base (Notion via Composio)

In [ ]:
from langgraph.prebuilt import create_react_agent

KB_PROMPT = '''You are the knowledge-base specialist. Given a support ticket, query the Notion KB
to find relevant docs. Summarise the top 1-2 hits in 3 sentences.
If nothing relevant exists, say "NO_KB_MATCH".'''

kb_agent = create_react_agent(llm, notion_tools, prompt=KB_PROMPT)

def kb_worker(state: TicketState) -> dict:
    query = f"Find docs about: {state['intent']} - {state['ticket_subject']}"
    result = kb_agent.invoke({"messages": [HumanMessage(content=query)]})
    excerpt = result["messages"][-1].content
    return {
        "kb_excerpt": excerpt,
        "messages": [AIMessage(content=f"KB result: {excerpt[:200]}", name="kb")],
    }


## 6. Worker C - Reply drafter (Gmail via Composio)

In [ ]:
REPLY_PROMPT = '''You are a customer support reply drafter. Use the KB excerpt to write a
helpful, friendly, brief reply. Sign off with "Support Team".

If the KB excerpt is "NO_KB_MATCH", write a holding reply that says we are looking into it
and will follow up within 24 hours.'''

def reply_worker(state: TicketState) -> dict:
    composer = llm.invoke([
        SystemMessage(content=REPLY_PROMPT),
        HumanMessage(content=(
            f"Ticket: {state['ticket_subject']}\n"
            f"Body: {state['ticket_body']}\n"
            f"Intent: {state['intent']}\n"
            f"KB excerpt: {state.get('kb_excerpt', 'NONE')}"
        )),
    ])
    draft = composer.content
    # decide whether to escalate before send
    needs_human = state["severity"] == "high" or "NO_KB_MATCH" in (state.get("kb_excerpt") or "")
    return {
        "draft_reply": draft,
        "needs_human": needs_human,
        "messages": [AIMessage(content=f"Draft prepared (needs_human={needs_human})", name="drafter")],
    }


## 7. Worker D - Sender / Escalator

If low-risk: send via Gmail. If high-risk: post draft to Slack with HITL interrupt.

In [ ]:
from langgraph.types import interrupt

slack_agent = create_react_agent(llm, slack_tools, prompt="Post the draft to #support-review channel for human approval.")
gmail_agent = create_react_agent(llm, gmail_tools, prompt="Send the draft as a reply to the original ticket.")

def sender_worker(state: TicketState) -> dict:
    if state["needs_human"]:
        # HITL pause - real run waits for human input
        approval = interrupt({
            "ticket_id": state["ticket_id"],
            "draft": state["draft_reply"],
            "prompt": "Approve send? Reply 'yes' or 'no'.",
        })
        if approval != "yes":
            # post to Slack instead of sending
            slack_agent.invoke({"messages": [HumanMessage(content=(
                f"Escalation needed for ticket {state['ticket_id']}.\n"
                f"Draft:\n{state['draft_reply']}"
            ))]})
            return {"messages": [AIMessage(content="Escalated to #support-review", name="sender")]}
    # auto-send path
    gmail_agent.invoke({"messages": [HumanMessage(content=(
        f"Reply to ticket {state['ticket_id']} with this body:\n{state['draft_reply']}"
    ))]})
    return {"messages": [AIMessage(content="Reply sent", name="sender")]}


## 8. Supervisor

Linear flow for this project: classify -> KB lookup -> draft -> send. The supervisor is essentially a state-machine sequencer.

In [ ]:
def supervisor(state: TicketState) -> dict:
    if state.get("intent") is None:
        return {"next_worker": "classifier"}
    if state.get("kb_excerpt") is None:
        return {"next_worker": "kb"}
    if state.get("draft_reply") is None:
        return {"next_worker": "drafter"}
    # all info gathered - send
    if "Reply sent" not in (state["messages"][-1].content if state["messages"] else ""):
        return {"next_worker": "sender"}
    return {"next_worker": "DONE"}

def route(state: TicketState) -> str:
    nxt = state["next_worker"]
    return nxt if nxt in {"classifier", "kb", "drafter", "sender"} else "__end__"


## 9. Build, compile, persist

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(TicketState)
g.add_node("supervisor", supervisor)
g.add_node("classifier", classifier_worker)
g.add_node("kb",         kb_worker)
g.add_node("drafter",    reply_worker)
g.add_node("sender",     sender_worker)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "classifier": "classifier", "kb": "kb",
    "drafter": "drafter", "sender": "sender",
    "__end__": END,
})
for w in ("classifier", "kb", "drafter", "sender"):
    g.add_edge(w, "supervisor")

conn = sqlite3.connect("support_resolver.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 10. Graph diagram (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 11. End-to-end demo run

A fake low-severity ticket so it auto-resolves without HITL.

In [ ]:
config = {"configurable": {"thread_id": "ticket-1001"}, "recursion_limit": 30}

initial = {
    "ticket_id": "1001",
    "ticket_subject": "How do I reset my password?",
    "ticket_body": "Hi, I forgot my password. Can you help me reset it? Thanks, Alice",
    "messages": [HumanMessage(content="New ticket arrived: How do I reset my password?")],
    "needs_human": False,
}

result = app.invoke(initial, config=config)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != "messages":
        print(f"{k}: {str(v)[:120]}")
print("\n=== MESSAGE TRACE ===")
for m in result["messages"]:
    label = getattr(m, "name", m.type)
    print(f"[{label}] {m.content[:200]}")


## 12. Submission checklist
- [x] 1 supervisor + 4 worker nodes (classifier, kb, drafter, sender)
- [x] All workers using Composio toolkits (notion + gmail + slack via Action enums)
- [x] HITL on destructive action (sender_worker uses `interrupt()` when severity=high)
- [x] Checkpoint persistence (SqliteSaver -> support_resolver.db)
- [x] Graph diagram (cell 10)
- [x] One example end-to-end run (cell 11)
- [ ] README explaining architecture, state object, example run -> write README.md next

## Files to ship
- `support_resolver.ipynb` (this notebook)
- `graph.png` (auto-generated cell 10)
- `README.md` (architecture + example)
- `support_resolver.db` (checkpoint sample)
